# Negotiation Environment — GRPO Training & Evaluation

**실행 순서:** 셀 1~8을 순서대로 실행하세요.

| 셀 | 내용 |
|---|---|
| 1 | GPU 확인 |
| 2 | 패키지 설치 |
| 3 | HuggingFace 토큰 & 레포 설정 |
| 4 | Google Drive 마운트 (LoRA 체크포인트 + 평가 결과 영구 저장) |
| 5 | GRPO 학습 |
| 6 | 평가 (학습 모델 vs 베이스라인) |
| 7 | 학습 곡선 시각화 |
| 8 | 평가 결과 비교표 |

**Drive 저장 구조 (총 ~300MB 수준):**
```
MyDrive/
  negotiation_grpo/
    checkpoints/          ← GRPO LoRA 가중치 (~200MB, 50스텝마다 자동 저장)
      step_00050/ ...
      final/              ← 최종 학습 모델
      training_log.jsonl  ← 스텝별 loss / deal_rate / reward
    eval_results/         ← 평가 결과 JSON + 비교 리포트 (~수 MB)
```

> **베이스 모델(Llama-3.1-8B, ~16GB)은 Drive에 저장하지 않습니다.**  
> 세션 시작 시 HuggingFace에서 재다운로드 (~3~5분, Colab 고속 인터넷 기준).  
> Drive에 캐싱하면 16GB를 차지하므로 비효율적입니다.

In [ ]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout)
# A100 40GB 권장. T4(16GB)의 경우 셀 5에서 --use_4bit 플래그 추가 필요.

In [ ]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────────
# 설치 후 런타임 재시작 불필요 (Colab 환경에서 즉시 적용)
%pip install -q \
    torch>=2.2.0 \
    transformers>=4.43.0 \
    peft>=0.11.0 \
    accelerate>=0.30.0 \
    bitsandbytes>=0.43.0 \
    openai>=1.30.0 \
    pandas \
    python-dotenv
print('설치 완료')

In [ ]:
# ── 셀 3: HuggingFace 토큰 & 레포 설정 ──────────────────────────
# Llama-3.1-8B 접근을 위해 HF 토큰 필요:
#   1. https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct 에서 라이선스 동의
#   2. https://huggingface.co/settings/tokens 에서 Read 토큰 발급
#   3. 아래 HF_TOKEN에 붙여넣기

import os
from google.colab import userdata

# Colab Secrets에 HF_TOKEN을 저장한 경우 (왼쪽 사이드바 🔑 아이콘):
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = "hf_여기에_토큰_붙여넣기"  # 직접 입력 시

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# 레포 클론
REPO_URL = "https://github.com/Azalea129/negotiation-env.git"
BRANCH   = "claude/negotiation-environment-review-6MoJP"
REPO_DIR = "/content/negotiation-env"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
print('작업 디렉토리:', os.getcwd())

In [ ]:
# ── 셀 4: Google Drive 마운트 ────────────────────────────────────
# LoRA 가중치(~200MB)와 평가 결과만 Drive에 저장.
# 베이스 Llama 모델(16GB)은 세션마다 HF에서 재다운로드 (~3~5분).

from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/negotiation_grpo/checkpoints'
EVAL_DIR = '/content/drive/MyDrive/negotiation_grpo/eval_results'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print(f'체크포인트: {CKPT_DIR}')
print(f'평가 결과:  {EVAL_DIR}')

# 이전 학습 체크포인트 확인
import os as _os
ckpts = sorted([d for d in _os.listdir(CKPT_DIR) if d.startswith('step_')]) if _os.path.exists(CKPT_DIR) else []
if ckpts:
    print(f'\n✓ 기존 체크포인트 발견: {ckpts[-1]} (마지막 저장)')
else:
    print('\n※ 첫 실행 — 체크포인트 없음')

In [ ]:
# ── 셀 5: GRPO 학습 ──────────────────────────────────────────────
#
# A100 40GB: 기본 설정 (bfloat16, 약 34GB 사용)
# T4  16GB:  --use_4bit 추가 (QLoRA, 약 12GB 사용)
#
# 예상 시간: A100 기준 500스텝 ≈ 5~8시간
# 빠른 테스트: --n_steps 20 --group_size 4 로 먼저 확인

import subprocess

# GPU 타입 자동 감지
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
use_4bit = 'T4' in gpu_info or '3090' in gpu_info or '2080' in gpu_info
print(f'GPU: {gpu_info}  →  use_4bit={use_4bit}')

cmd = [
    'python', 'train_grpo.py',
    '--model_id',    'meta-llama/Meta-Llama-3.1-8B-Instruct',
    '--n_steps',     '500',
    '--group_size',  '8',
    '--max_rounds',  '10',
    '--temperature', '0.8',
    '--lambda_s',    '0.5',
    '--condition',   'free_form',
    '--lr',          '1e-5',
    '--clip_eps',    '0.2',
    '--kl_beta',     '0.01',
    '--lora_r',      '16',
    '--lora_alpha',  '32',
    '--output_dir',  CKPT_DIR,
    '--save_every',  '50',
    '--log_every',   '10',
]
if use_4bit:
    cmd.append('--use_4bit')

print('\n실행 명령:\n', ' '.join(cmd), '\n')
!{' '.join(cmd)}

In [ ]:
# ── 셀 6: 평가 ───────────────────────────────────────────────────
# 학습 완료 후 실행. 세팅별 100 에피소드 × 4 매칭 조건 = 1,200 에피소드
# 예상 시간: A100 기준 ≈ 2~3시간

TRAINED_PATH = f'{CKPT_DIR}/final'

cmd_eval = [
    'python', 'evaluate.py',
    '--base_model_id', 'meta-llama/Meta-Llama-3.1-8B-Instruct',
    '--trained_path',  TRAINED_PATH,
    '--n_episodes',    '100',
    '--n_parties',     '3',
    '--max_rounds',    '10',
    '--condition',     'free_form',
    '--lambda_s',      '0.5',
    '--settings',      '1v1', '1vn', 'nv1',
    '--output_dir',    EVAL_DIR,
]
print('\n실행 명령:\n', ' '.join(cmd_eval), '\n')
!{' '.join(cmd_eval)}

In [ ]:
# ── 셀 7: 학습 곡선 시각화 ──────────────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

log_path = f'{CKPT_DIR}/training_log.jsonl'
logs = [json.loads(l) for l in open(log_path)]

steps        = [l['step'] for l in logs]
losses       = [l['loss'] for l in logs]
deal_rates   = [l['deal_rate'] for l in logs]
buyer_rews   = [l['avg_buyer_reward'] for l in logs]
seller_rews  = [l['avg_seller_reward'] for l in logs]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, losses, color='steelblue')
axes[0].set_title('GRPO Loss'); axes[0].set_xlabel('Step')

axes[1].plot(steps, deal_rates, color='seagreen')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('Agreement Rate per Group'); axes[1].set_xlabel('Step')

axes[2].plot(steps, buyer_rews,  label='Buyer',  color='royalblue')
axes[2].plot(steps, seller_rews, label='Seller', color='tomato')
axes[2].set_title('Avg Reward'); axes[2].set_xlabel('Step')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curve.png', dpi=150)
plt.show()
print('학습 곡선 저장 완료')

In [ ]:
# ── 셀 8: 평가 결과 비교표 ──────────────────────────────────────
import json, os
import pandas as pd

summary_path = f'{EVAL_DIR}/summary.json'
with open(summary_path) as f:
    summary = json.load(f)

MATCH_LABELS = {
    'GG': 'GRPO vs GRPO',
    'GB': 'GRPO buyer / Base seller',
    'BG': 'Base buyer / GRPO seller',
    'BB': 'Base vs Base',
}
METRICS = {
    'agreement_rate':         'Agreement Rate',
    'avg_deal_round':         'Avg Rounds to Deal',
    'avg_total_welfare_all':  'Avg Welfare (all ep)',
    'avg_buyer_reward':       'Avg Buyer Reward',
    'avg_seller_reward':      'Avg Seller Reward',
    'efficiency':             'Efficiency',
}

for setting, cond_data in summary.items():
    rows = []
    for mc, stats in cond_data.items():
        row = {'Condition': MATCH_LABELS.get(mc, mc)}
        for key, label in METRICS.items():
            val = stats.get(key, 0)
            if 'rate' in key or 'efficiency' in key:
                row[label] = f'{val:.1%}'
            else:
                row[label] = f'{val:.2f}'
        rows.append(row)

    df = pd.DataFrame(rows).set_index('Condition')
    print(f'\n── Setting: {setting} ──')
    display(df)

    # GRPO 우위 요약
    if 'GB' in cond_data and 'BB' in cond_data:
        gb_br = cond_data['GB']['avg_buyer_reward']
        bb_br = cond_data['BB']['avg_buyer_reward']
        bg_sr = cond_data.get('BG', {}).get('avg_seller_reward', 0)
        bb_sr = cond_data['BB']['avg_seller_reward']
        print(f'  → GRPO Buyer  advantage: Δ={gb_br - bb_br:+.3f}')
        print(f'  → GRPO Seller advantage: Δ={bg_sr - bb_sr:+.3f}')